# 🍛 Food Detection - Version 1 Model Training (YOLO)
**South Indian Food Nutrition & Detection Project**

This notebook walks through the complete end-to-end training, validation, evaluation, and inference pipeline for the **Version 1 Food Detection Model** using Ultralytics YOLOv8 / YOLO11 on **31 South Indian food categories** (8,010 real-world annotated images).

---

### 📋 Workflow Overview:
1. **Environment Setup & Dependency Check**: Confirm GPU runtime and YOLO dependencies.
2. **Dataset Configuration & Inspection**: Load `dataset.yaml`, inspect splits (6,403 train, 812 val, 795 test), and verify 31 classes.
3. **Data Visualization**: Inspect sample training images with ground-truth bounding boxes.
4. **Model Architecture Selection**: Load pretrained `yolov8s.pt` (or `yolo11s.pt`) weights.
5. **Model Training**: Train the detector with optimized hyperparameters (640x640, AdamW/auto, early stopping).
6. **Training Metrics Analysis**: Visualize loss curves, mAP50, mAP50-95, precision, recall, and confusion matrix.
7. **Test Set Evaluation**: Benchmark performance on the held-out 795 test images.
8. **Visual Inference on Test Food Plates**: Run detection on unseen food photos and display prediction overlays.
9. **Model Export & Deployment**: Export `best.pt` to ONNX format for edge deployment.

## Step 1: Environment & Dependency Setup
We install `ultralytics` and verify that a GPU is available for deep learning training.

In [ ]:
# Install Ultralytics and dependencies
!pip install -q ultralytics torchvision opencv-python-headless pyyaml matplotlib seaborn

import torch
import ultralytics

print(f"PyTorch Version   : {torch.__version__}")
print(f"Ultralytics       : {ultralytics.__version__}")
print(f"CUDA Available    : {'✅ Yes' if torch.cuda.is_available() else '❌ No (Switch Colab runtime to GPU!)'}")
if torch.cuda.is_available():
    print(f"Device Name       : {torch.cuda.get_device_name(0)}")

# Run Ultralytics checks
ultralytics.checks()

## Step 2: Project Repository & Dataset Setup
Ensure you have cloned the project repository or mounted Google Drive containing the `Version1 Used Data` folder.
* **Train split**: 6,403 images
* **Validation split**: 812 images
* **Test split**: 795 images
* **Classes (31)**: All popular South Indian dishes including Idly, Dosa, Chicken briyani, Mutton Briyani, Sambar, Ven Pongal, etc.

In [ ]:
import os

# Clone repo if in Google Colab and not yet cloned
REPO_URL = "https://github.com/Dharmenthiran-spi/SouthIndian_Food_Nutrition.git"
REPO_DIR = "SouthIndian_Food_Nutrition"

if not os.path.exists(REPO_DIR) and not os.path.exists("Food_Detection"):
    !git clone {REPO_URL}
    if os.path.exists(REPO_DIR):
        os.chdir(REPO_DIR)

print(f"Current Working Directory: {os.getcwd()}")

### Locate and Validate `dataset.yaml`
We load `dataset.yaml` for Version 1 Used Data and inspect all 31 food classes.

In [ ]:
import yaml

# Candidate paths for dataset.yaml
possible_paths = [
    "Food_Detection/DataSet/Version1 Used Data/dataset.yaml",
    "/content/SouthIndian_Food_Nutrition/Food_Detection/DataSet/Version1 Used Data/dataset.yaml",
    "D:/Yolo_Food_Inspection/Food_Detection/DataSet/Version1 Used Data/dataset.yaml"
]

dataset_yaml_path = None
for p in possible_paths:
    if os.path.exists(p):
        dataset_yaml_path = os.path.abspath(p)
        break

if dataset_yaml_path:
    print(f"✅ Found dataset.yaml at: {dataset_yaml_path}")
    with open(dataset_yaml_path, 'r', encoding='utf-8') as f:
        data_cfg = yaml.safe_load(f)
    
    print(f"Total Classes ({data_cfg.get('nc', len(data_cfg.get('names', [])))}):")
    names = data_cfg.get('names', {})
    if isinstance(names, dict):
        for idx, name in sorted(names.items()):
            print(f"  {idx:2d}: {name}")
    elif isinstance(names, list):
        for idx, name in enumerate(names):
            print(f"  {idx:2d}: {name}")
else:
    print("⚠️ dataset.yaml not found at expected local path.")
    print("Please set `dataset_yaml_path` to the correct path of your dataset.yaml.")

## Step 3: Ground-Truth Inspection & Data Visualization
Before training, it is best practice to visualize a sample of training images and verify that bounding boxes align accurately with food items.

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Find dataset directory
if dataset_yaml_path:
    dataset_dir = os.path.dirname(dataset_yaml_path)
    train_img_dir = os.path.join(dataset_dir, "images", "train")
    train_lbl_dir = os.path.join(dataset_dir, "labels", "train")
    
    img_files = glob.glob(os.path.join(train_img_dir, "*.jpg"))[:4]
    
    if img_files:
        fig, axes = plt.subplots(1, min(4, len(img_files)), figsize=(18, 5))
        if len(img_files) == 1:
            axes = [axes]
            
        for ax, img_path in zip(axes, img_files):
            img = cv2.imread(img_path)
            h, w, _ = img.shape
            
            # Read matching label
            stem = os.path.splitext(os.path.basename(img_path))[0]
            lbl_path = os.path.join(train_lbl_dir, f"{stem}.txt")
            
            if os.path.exists(lbl_path):
                with open(lbl_path, 'r') as lf:
                    for line in lf:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            cid = int(parts[0])
                            xc, yc, bw, bh = map(float, parts[1:5])
                            x1 = int((xc - bw / 2) * w)
                            y1 = int((yc - bh / 2) * h)
                            x2 = int((xc + bw / 2) * w)
                            y2 = int((yc + bh / 2) * h)
                            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                            cname = names[cid] if isinstance(names, dict) and cid in names else str(cid)
                            cv2.putText(img, cname, (x1, max(y1 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(stem[:20], fontsize=10)
            ax.axis('off')
            
        plt.tight_layout()
        plt.show()
    else:
        print("No sample images found to display.")

## Step 4: Model Selection & Pretrained Initialization
For Food Detection Version 1, we use `yolov8s.pt` (YOLOv8 Small) or `yolo11s.pt` (YOLO11 Small).
* **Speed vs Accuracy**: The `s` (small) model achieves ~45-50 mAP on food datasets while maintaining 100+ FPS on edge devices.
* **Transfer Learning**: Loading pretrained COCO weights accelerates convergence and prevents overfitting.

In [ ]:
from ultralytics import YOLO

# Select model architecture: yolov8s.pt, yolov8n.pt, or yolo11s.pt
MODEL_WEIGHTS = 'yolov8s.pt'

print(f"📥 Loading pretrained {MODEL_WEIGHTS}...")
model = YOLO(MODEL_WEIGHTS)
print("✅ Model successfully initialized.")

## Step 5: Train the Food Detection Version 1 Model
### Automatic Checkpointing (`best.pt` & `last.pt`):
* `save=True`: Instructs Ultralytics YOLO to automatically save **both** checkpoints:
  * **`best.pt`**: Updated whenever validation metrics (mAP50-95) reach a new high score. Ideal for production and deployment.
  * **`last.pt`**: Updated at the end of **every single epoch**. Contains model state, optimizer state, and epoch number so you can resume training if disconnected.
* `save_period=10`: Saves a milestone checkpoint every 10 epochs (`epoch10.pt`, `epoch20.pt`, etc.).
* `imgsz`: 640x640 resolution.
* `batch`: 16 (adjust to 8 if encountering GPU Out-Of-Memory).
* `patience`: Early stopping threshold (stops if validation mAP does not improve for 20 epochs).

Run the cell below to start training:

In [ ]:
# Training Configuration - Guaranteed best.pt & last.pt saving
EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 640
PROJECT = "Food_Detection_V1"
EXPERIMENT_NAME = "yolov8s_31classes"

print(f"🚀 Starting training on {dataset_yaml_path}...")
results = model.train(
    data=dataset_yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=20,
    save=True,          # Ensures best.pt AND last.pt are saved at every epoch
    save_period=10,     # Saves milestone checkpoint every 10 epochs
    project=PROJECT,
    name=EXPERIMENT_NAME,
    exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    verbose=True
)

print("\n🎉 Training Complete!")
print(f"Checkpoints saved at: {PROJECT}/{EXPERIMENT_NAME}/weights/")

## Step 6: Review Training Performance & Curves
Ultralytics automatically logs loss curves, mAP metrics, and confusion matrices in the run directory.
Let's visualize the summary `results.png` and `confusion_matrix.png`.

In [ ]:
from IPython.display import Image, display

RUN_DIR = os.path.join(PROJECT, EXPERIMENT_NAME)
results_img = os.path.join(RUN_DIR, "results.png")
confusion_img = os.path.join(RUN_DIR, "confusion_matrix.png")

if os.path.exists(results_img):
    print("📈 Training Loss & Validation mAP Curves:")
    display(Image(filename=results_img, width=900))
else:
    print(f"results.png not found at {results_img}")

if os.path.exists(confusion_img):
    print("🎯 Confusion Matrix:")
    display(Image(filename=confusion_img, width=900))

## Step 7: Evaluate Model on Held-Out Test Set
We benchmark the trained weights (`best.pt`) on the 795 test images to measure generalization performance.

In [ ]:
best_model_path = os.path.join(RUN_DIR, "weights", "best.pt")

if os.path.exists(best_model_path):
    print(f"Loading best checkpoint from: {best_model_path}")
    best_model = YOLO(best_model_path)
    
    print("\nEvaluating on test split...")
    metrics = best_model.val(split='test')
    
    print("\n==========================================")
    print("📊 TEST SET METRICS:")
    print("==========================================")
    print(f"mAP@0.50     : {metrics.box.map50:.4f}")
    print(f"mAP@0.50:0.95: {metrics.box.map:.4f}")
    print(f"Precision    : {metrics.box.mp:.4f}")
    print(f"Recall       : {metrics.box.mr:.4f}")
    print("==========================================")
else:
    print(f"best.pt not found at {best_model_path}")

## Step 8: Visual Inference on Test Images
Now we run predictions on sample test images and visualize the detected food items with bounding boxes, confidence scores, and class names.

In [ ]:
if os.path.exists(best_model_path) and dataset_yaml_path:
    test_img_dir = os.path.join(dataset_dir, "images", "test")
    sample_tests = glob.glob(os.path.join(test_img_dir, "*.jpg"))[:3]
    
    if sample_tests:
        preds = best_model.predict(source=sample_tests, conf=0.25, iou=0.45)
        
        fig, axes = plt.subplots(1, len(sample_tests), figsize=(18, 6))
        if len(sample_tests) == 1:
            axes = [axes]
            
        for ax, pred in zip(axes, preds):
            annotated_img = pred.plot()
            ax.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
            ax.set_title("Detection Output", fontsize=12)
            ax.axis('off')
            
        plt.tight_layout()
        plt.show()

## Step 9: Save & Backup `best.pt` and `last.pt` Models
This step automatically backs up **both** `best.pt` and `last.pt` models:
1. **`best.pt`**: The top-performing model for inference and production.
2. **`last.pt`**: The latest epoch checkpoint, allowing you to resume training seamlessly at any time.
3. Exporting to **ONNX** format for deployment.
4. Copies all models to the local project folder and Google Drive.

In [ ]:
import os
import shutil

weights_dir = os.path.join(PROJECT, EXPERIMENT_NAME, "weights")
best_pt = os.path.join(weights_dir, "best.pt")
last_pt = os.path.join(weights_dir, "last.pt")

# Target directories to save both models
targets = [
    "Food_Detection/Models/Food Detection/Version1",
    "/content/drive/MyDrive/SouthIndian_Food_Nutrition/Food_Detection/Models/Food Detection/Version1"
]

print("=" * 60)
print("📦 SAVING & BACKING UP best.pt AND last.pt MODELS")
print("=" * 60)

# Check and display sizes of both checkpoints
for name, path in [("BEST MODEL (best.pt)", best_pt), ("LAST MODEL (last.pt)", last_pt)]:
    if os.path.exists(path):
        sz_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {name:<22}: {sz_mb:.2f} MB -> Found at {path}")
    else:
        print(f"❌ {name:<22}: Not found at {path}")

# Copy both models to local and Google Drive destinations
for target_dir in targets:
    if os.path.exists(os.path.dirname(target_dir)) or target_dir.startswith("Food_Detection"):
        os.makedirs(target_dir, exist_ok=True)
        if os.path.exists(best_pt):
            shutil.copyfile(best_pt, os.path.join(target_dir, "best.pt"))
            print(f"   -> Saved best.pt to: {target_dir}/best.pt")
        if os.path.exists(last_pt):
            shutil.copyfile(last_pt, os.path.join(target_dir, "last.pt"))
            print(f"   -> Saved last.pt to: {target_dir}/last.pt")

# Export ONNX format from best model
if os.path.exists(best_pt):
    print("\n📦 Exporting best.pt to ONNX format for deployment...")
    onnx_file = best_model.export(format='onnx', dynamic=True, simplify=True)
    for target_dir in targets:
        if os.path.exists(target_dir) and os.path.exists(str(onnx_file)):
            shutil.copyfile(str(onnx_file), os.path.join(target_dir, "best.onnx"))
            print(f"   -> Saved best.onnx to: {target_dir}/best.onnx")

print("=" * 60)
print("🎉 All models (best.pt, last.pt, best.onnx) successfully secured!")
print("=" * 60)

## Step 10: How to Resume Training from `last.pt` (If Interrupted)
If your Google Colab session disconnects or reaches maximum runtime limits, you **never lose your progress**.
Simply run the cell below to pick up training exactly where it stopped using `last.pt`:

In [ ]:
# Resume training if interrupted
from ultralytics import YOLO
import os

LAST_CHECKPOINT = "Food_Detection/Models/Food Detection/Version1/last.pt"
if not os.path.exists(LAST_CHECKPOINT):
    LAST_CHECKPOINT = os.path.join(PROJECT, EXPERIMENT_NAME, "weights", "last.pt")

if os.path.exists(LAST_CHECKPOINT):
    print(f"🔄 Resuming training from checkpoint: {LAST_CHECKPOINT}")
    resume_model = YOLO(LAST_CHECKPOINT)
    resume_results = resume_model.train(resume=True)
    print("🎉 Training resumed and completed successfully!")
else:
    print(f"ℹ️ No last.pt checkpoint found at {LAST_CHECKPOINT} to resume.")

## 🏁 Summary
Congratulations! You have completed:
* Verified environment and dataset configuration
* Trained YOLOv8s on 31 South Indian food classes
* Evaluated precision, recall, and mAP50 on the test split
* Exported the final production-ready model weights (`best.pt` & `best.onnx`)

### Next Steps:
* Deploy the model into your nutrition estimation and calorie calculator pipeline.
* Integrate with the **Ingredient Identification** module in `Incredient_Identification`!